By default, 40 example generations are recorded during the test run.

This is a simple notebook that allows you to view more generations (and reconstructions).

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.vae.info_adversarial_vae import InfoAdversarialVAE
from arch.vae.info_vae import InfoVAE
from arch.vae.vae import VAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

# model_path = "logs/vae_acdc/info-adversarial-vae/ld-16-beta-0-gamma-50/checkpoints/epoch=44-step=9630.ckpt"
# model_path = "logs/vae_acdc/info-vae/ld-8-beta-1-gamma-100/checkpoints/epoch=47-step=5136.ckpt"

model_path = "logs/vae_acdc/best/beta-vae/checkpoints/epoch=47-step=5136.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = InfoVAE.load_from_checkpoint(model_path)

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader()
x = get_data(loader_test)
x.shape

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    x = x.to(device)
    
    num_data = x.shape[0]
    samples_idx = torch.randperm(num_data)[:24]
    x = x[samples_idx]
    _, _, _, x_hat_logits = model(x, test=True)

samples, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(x, x_hat_logits)
show_samples(samples, reconstruction_pixel_error, rgb=False, ncol=6, figsize=(24, 16), display=False)

In [ ]:
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)

    z = torch.randn(24, model.hparams.latent_dim).to(device)
    x_fake_logits: torch.Tensor = model.decoder(z)

# Discretise probabilistic map then view generations
generations = torch.argmax(x_fake_logits, dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=6, figsize=(24, 16))